# clipsmith — Twitch VOD → AI Vertical Clips (Colab T4)

**What this notebook does:**
1. Installs clipsmith and GPU transcription dependencies
2. Mounts your Google Drive for persistent storage
3. Downloads a Twitch VOD, transcribes it on the T4 GPU (~10 min for a 3-hr stream)
4. Uses an LLM to select the best 15–30 s clip moments
5. Cuts final clips with ffmpeg and lets you preview them inline

**LLM options (pick one):**
| Provider | Cost | Setup |
|----------|------|-------|
| **Ollama** (recommended) | Free | Run Cell 3a — downloads ~4.7 GB model to Colab once per session |
| **Anthropic Claude** | ~$0.01–0.05/VOD | Add `ANTHROPIC_API_KEY` to Colab Secrets (🔑 left panel) |

**Requirements:**
- Colab runtime: **GPU → T4** (Runtime → Change runtime type)
- For Anthropic: add `ANTHROPIC_API_KEY` to Colab Secrets
- For Twitch clip boost data (optional): add `TWITCH_CLIENT_ID` + `TWITCH_CLIENT_SECRET`

Each cell is re-runnable — cached transcripts and picks are loaded from disk on subsequent runs.

In [ ]:
# @title 1. Install clipsmith and GPU dependencies
# Takes ~2 min on first run; skip on subsequent runs if kernel is still alive.
!pip install -q "git+https://github.com/ricardogr07/clipsmith.git[ollama]"
# GPU-capable faster-whisper (CUDA 11.x, bundled with T4 Colab runtime)
!pip install -q faster-whisper nvidia-cublas-cu11 nvidia-cudnn-cu11
# ffmpeg is pre-installed on Colab; verify:
!ffmpeg -version | head -1

In [ ]:
# @title 2. Mount Google Drive and load API keys from Colab Secrets
# API key is only needed if using Anthropic (skip if using Ollama).
from google.colab import drive
import os

drive.mount("/content/drive")

# Anthropic key — only required if LLM_PROVIDER = 'anthropic' in Cell 4
try:
    from google.colab import userdata

    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY") or ""
except Exception:
    pass  # no key set — fine if using Ollama

# Optional Twitch credentials for existing-clip boost signals:
# os.environ['TWITCH_CLIENT_ID']     = userdata.get('TWITCH_CLIENT_ID')
# os.environ['TWITCH_CLIENT_SECRET'] = userdata.get('TWITCH_CLIENT_SECRET')

print("Drive mounted.")

In [ ]:
# @title 3a. (Ollama only) Install Ollama server and pull model
# Skip this cell entirely if you are using Anthropic.
# To persist model across sessions, uncomment the Drive symlink lines below.
import subprocess
import sys
import time

OLLAMA_MODEL = "llama3.1:8b"  # alternatives: mistral:7b (~4.1 GB), phi3:mini (~2.3 GB)

# --- Optional: persist models on Drive to avoid re-downloading each session ---
# import os
# os.makedirs("/content/drive/MyDrive/ollama_models", exist_ok=True)
# os.environ["OLLAMA_MODELS"] = "/content/drive/MyDrive/ollama_models"
# -------------------------------------------------------------------------------

print("Installing Ollama...")
subprocess.run(
    "curl -fsSL https://ollama.com/install.sh | sh",
    shell=True,
    check=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
print("Ollama installed.")

subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)  # let server bind before pull
print("Ollama server started.")

print(f"Pulling {OLLAMA_MODEL} - this takes a few minutes on first run...")
subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True)
subprocess.run([sys.executable, "-m", "clipsmith", "check-ollama"], check=True)

In [ ]:
# @title 4. Configure — set your VOD ID and LLM provider here
from pathlib import Path

VIDEO_ID = "2758856582"  # <-- paste your Twitch VOD ID
LLM_PROVIDER = "ollama"  # <-- 'ollama' (free) or 'anthropic'
OLLAMA_MODEL = "llama3.1:8b"  # ignored when LLM_PROVIDER = 'anthropic'
WORK_DIR = "/content/drive/MyDrive/clipsmith/work"
OUT_DIR = "/content/drive/MyDrive/clipsmith/out"

Path(WORK_DIR).mkdir(parents=True, exist_ok=True)
Path(OUT_DIR).mkdir(parents=True, exist_ok=True)

# Build LLM section based on provider choice
llm_section = f"""\
llm:
  provider: {LLM_PROVIDER}
  model_anthropic: claude-haiku-4-5-20251001
  model_ollama: {OLLAMA_MODEL}
"""

config_yaml = f"""\
work_dir: {WORK_DIR}
out_dir:  {OUT_DIR}

transcribe:
  model: medium          # medium balances speed/quality on T4
  compute_type: float16  # float16 = full T4 GPU precision
  language: es           # change to 'en' for English streams

{llm_section}
clip:
  min_seconds: 15
  max_seconds: 30
  preroll_s: 25
  postroll_s: 10

reframe:
  mode: none  # set to 'center' for 9:16 vertical crop

caption:
  enabled: false
"""

Path("config.yaml").write_text(config_yaml)
print(f"Config written — VOD: {VIDEO_ID}, LLM: {LLM_PROVIDER}")

In [ ]:
# @title 5. Download VOD and transcribe (T4 GPU)
# Transcription of a 3-hour stream takes ~10 min on T4.
# Re-running skips both steps if cached files already exist.
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "clipsmith",
        "process",
        VIDEO_ID,
        "--config",
        "config.yaml",
        "--skip-select",
        "--skip-clip",
    ],
    check=True,
)
print("Download + transcription complete.")

In [ ]:
# @title 6. LLM clip selection
# Uses Ollama (free, local on T4) or Anthropic depending on Cell 4 setting.
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "clipsmith",
        "process",
        VIDEO_ID,
        "--config",
        "config.yaml",
        "--skip-download",
        "--skip-transcribe",
        "--skip-chat",
        "--skip-clip",
    ],
    check=True,
)
print("Clip selection complete.")

In [ ]:
# @title 7. Cut final clips with ffmpeg
import subprocess
import sys

subprocess.run(
    [
        sys.executable,
        "-m",
        "clipsmith",
        "process",
        VIDEO_ID,
        "--config",
        "config.yaml",
        "--skip-download",
        "--skip-transcribe",
        "--skip-chat",
        "--skip-select",
    ],
    check=True,
)
print("Clips cut and saved to Drive.")

In [ ]:
# @title 8. Preview clips inline
from IPython.display import Video, display
from pathlib import Path

clip_dir = Path(OUT_DIR) / VIDEO_ID
clips = sorted(clip_dir.glob("clip_*.mp4"))

if not clips:
    print("No clips found. Did step 7 complete successfully?")
else:
    print(f"Found {len(clips)} clip(s):")
    for mp4 in clips:
        print(f"  {mp4.name}")
        display(Video(str(mp4), embed=True, width=360))